# Start Here — VSS Workshop

Deploy NVIDIA Video Search and Summarization (VSS) on this AWS Brev instance, then upload a short MP4, ask visual questions, and generate a report.

This workshop has one validated topology: **two RTX PRO 6000 GPUs**. GPU 0 runs NVIDIA Nemotron Nano 9B v2; GPU 1 runs Cosmos3 Nano Reasoner (Nano). Allow 15–25 minutes for the first model download.

## Goal and flow

1. Add your NGC API key privately.
2. Check the GPU, driver, Docker, storage, and Base deployment configuration.
3. Deploy VSS and wait for the UI.
4. Upload your own short MP4; then use visual Q&A and report generation.
5. Stop services safely when you are finished.

Your key is never written to Git or displayed by this notebook. The deployment command stores it only in a permission-restricted file on this VM so Docker can pull the NVIDIA containers and models.

## 1. Add your NGC API key

Create or find your key at [NGC API Keys](https://org.ngc.nvidia.com/setup/api-keys). Run the cell below and paste it only when prompted. Do not put the key in a notebook cell, chat, or screenshot.

In [ ]:
from getpass import getpass
from pathlib import Path
import os

def find_repo_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'workshop' / 'scripts' / 'deploy_vss_base.sh').is_file():
            return candidate
    raise FileNotFoundError('Open this notebook from the vss-workshop repository in Jupyter.')

REPO_ROOT = find_repo_root()
DEPLOY_SCRIPT = REPO_ROOT / 'workshop' / 'scripts' / 'deploy_vss_base.sh'
ngc_api_key = getpass('Paste your NGC API key (input is hidden): ').strip()
if not ngc_api_key:
    raise ValueError('An NGC API key is required for this workshop.')
os.environ['NGC_API_KEY'] = ngc_api_key
print('Key accepted. It is held in this running kernel and passed privately to the deployment command.')

## 2. Check this instance

This confirms that the VM has exactly two RTX PRO 6000 GPUs, a supported NVIDIA driver and Docker version, enough storage, and a valid Base Compose graph. If the current Brev image has a Docker version newer than the VSS-supported range, the check reports a managed repair; continue to the deploy step on this clean workshop VM. Fix every other failure before deploying.

In [ ]:
import subprocess

subprocess.run([str(DEPLOY_SCRIPT), 'check'], check=True)

## 3. Deploy VSS

The command pins an unsupported Docker CE engine to the newest VSS-compatible 28.x release when needed, then prepares the workshop data directory and required Redis/VIOS writable paths on large ephemeral storage when available. If the existing Docker filesystem is too small, it will ask for `sudo` to move Docker storage and restart Docker on this clean workshop VM. It signs in to NVCR through password stdin and records detailed image-download progress in a local deployment log rather than flooding this notebook. It then starts the Base services, waits for health checks, and prints the UI address.

The first run can take 15–25 minutes while the model images and weights initialize. Keep this notebook cell running.

In [ ]:
subprocess.run([str(DEPLOY_SCRIPT), 'deploy'], check=True, env=os.environ.copy())

## 4. Open the UI and check health

Use the URL printed above. In Brev, open or create a **secure link for port 7777 only**; use that secure-link URL if it differs from the printed address. The status command below is safe to run at any time.

In [ ]:
subprocess.run([str(DEPLOY_SCRIPT), 'status'], check=True)

## 5. Upload, ask, and report

1. The **VSS Agent** chat panel stays open on the left. On the right, use **Video Management** to upload your own short MP4. Wait for the upload/processing indicator to finish.
2. Click **+ Chat** on the video card to attach that video to the open VSS Agent conversation, then try: `Give me a concise timeline of what happens in this video.`
3. Ask a focused visual question, such as: `At what timestamp does the main action begin?`
4. Generate a report, keeping the default prompt or revising it to focus on people, objects, or safety-relevant events.

Use a short H.264 MP4 for the best workshop experience.

## 6. Stop safely when finished

The command below stops VSS but does **not** delete uploaded videos, Docker volumes, model caches, or the VM-local deployment state. Afterward, stop the Brev VM to stop GPU billing. Only terminate the VM after you no longer need its data.

In [ ]:
stop_vss_now = False  # Change to True only when you are ready to stop the workshop services.
if stop_vss_now:
    subprocess.run([str(DEPLOY_SCRIPT), 'stop'], check=True)
else:
    print('VSS is still running. Change stop_vss_now to True when you are finished.')